# A Knowledge-Guided Hybrid Transformer for Early Sepsis Prediction
## Google Colab Automated Execution Pipeline

This notebook automates the execution of the M2 (Plain Transformer) model on Google Colab GPU. It strictly follows the constraints to NOT modify methodology, preprocessing, or architecture.

### PHASE 1: GOOGLE COLAB SETUP

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/Sepsis-Hybrid-Transformer'
if not os.path.exists(BASE_DIR):
    os.makedirs(BASE_DIR)

dirs_to_create = ['data', 'processed', 'checkpoints', 'experiments', 'logs', 'plots', 'figures', 'code']
for d in dirs_to_create:
    os.makedirs(os.path.join(BASE_DIR, d), exist_ok=True)

print("\n[Phase 1] Drive mounted and directory structure verified.")

### PHASE 2: GITHUB SYNCHRONIZATION

In [ ]:
import subprocess
import sys

CODE_DIR = os.path.join(BASE_DIR, 'code')
os.chdir(BASE_DIR)

# Note: Replace <repository_url> with your actual GitHub URL if cloning for the first time
REPO_URL = "<repository_url>"

if os.path.exists(os.path.join(CODE_DIR, ".git")):
    print("Repository found. Pulling latest changes...")
    os.chdir(CODE_DIR)
    !git pull
else:
    print("Repository not found. Cloning...")
    !git clone {REPO_URL} code
    os.chdir(CODE_DIR)

print("\n[Phase 2] GitHub synchronization complete.")

### PHASE 3: VERIFY ENVIRONMENT

In [ ]:
import sys
import torch

print("=== ENVIRONMENT VERIFICATION ===")
print(f"Python version: {sys.version.split(' ')[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise SystemError("GPU unavailable. Reconnect to a GPU runtime. Do not continue on CPU.")

print(f"CUDA Version: {torch.version.cuda}")
print(f"GPU Name: {torch.cuda.get_device_name(0)}")
mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU Memory: {mem_gb:.2f} GB")
print("\n=== INSTALLED PACKAGES ===")
!pip list | grep -E "numpy|pandas|matplotlib|scikit-learn|tensorboard|torchvision|xgboost"

print("\n[Phase 3] Environment verification successful.")

### PHASE 4: VERIFY DATASET

In [ ]:
outer_cache = os.path.join(BASE_DIR, "processed", "full_dataset_cache.pt")
inner_dir = os.path.join(CODE_DIR, "data", "processed")
inner_cache = os.path.join(inner_dir, "full_dataset_cache.pt")

cache_target = None
if os.path.exists(inner_cache):
    cache_target = inner_cache
elif os.path.exists(outer_cache):
    cache_target = outer_cache
else:
    raise FileNotFoundError(f"full_dataset_cache.pt missing! Please upload the 1.3GB cache file to:\n{outer_cache}\nOR\n{inner_cache}\nSTOP. Never regenerate preprocessing.")
    
print(f"Loading cached dataset from {cache_target}...")
cache = torch.load(cache_target)

splits = {'train': 0, 'val': 0, 'test': 0}
for k, v in cache.items(): 
    splits[v['split']] += 1

print("\n=== DATASET VERIFICATION ===")
print(f"Dataset exists and is readable.")
print(f"Total dataset size: {len(cache)} patients")
print(f"Train split: {splits['train']}")
print(f"Validation split: {splits['val']}")
print(f"Test split: {splits['test']}")

sample_pid = list(cache.keys())[0]
print(f"Tensor dimensions (Values): {cache[sample_pid]['values'].shape}")
print(f"Tensor dimensions (Mask): {cache[sample_pid]['mask'].shape}")

print("\n[Phase 4] Dataset verification successful.")

### PHASE 5: VERIFY TRAINING PIPELINE (100 Batches)

In [ ]:
os.chdir(CODE_DIR)
!python scripts/train.py --config configs/m2.yaml --verify

### PHASE 6-12: FULL TRAINING, LIVE MONITORING & EVALUATION
Launch TensorBoard in the background to monitor metrics live, then execute the full training pipeline with automatic resuming capability.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir experiments/

In [ ]:
os.chdir(CODE_DIR)
!python scripts/train.py --config configs/m2.yaml --resume